# Strace log → table

Parses `out.log` (strace-style lines) with **pandas**, keeps only lines that reference paths under `/home/anurag/opensource/TEST/`, and shows the result as a table.


In [ ]:
import re
from pathlib import Path

import pandas as pd

DEFAULT_PATH_PREFIX = "/home/anurag/opensource/TEST/"

_ERRNO_RE = re.compile(r"^(-?\d+)\s+([A-Z0-9_]+)\s+\((.*)\)\s*$")
_INT_RE = re.compile(r"^-?\d+$")


def _split_syscall_and_args(left):
    try:
        i = left.index("(")
    except ValueError:
        return None, None
    name = left[:i].strip()
    depth = 0
    for j in range(i, len(left)):
        c = left[j]
        if c == "(":
            depth += 1
        elif c == ")":
            depth -= 1
            if depth == 0:
                return name, left[i + 1 : j]
    return name, left[i + 1 :]


def parse_strace_line(line):
    line = line.rstrip("\n\r")
    row: dict = {"raw": line}

    if " = " not in line:
        row["kind"] = "message" if line.strip() else "blank"
        row["syscall"] = None
        row["arguments"] = None
        row["result_raw"] = None
        row["errno"] = None
        row["errno_message"] = None
        return row

    left, right = line.rsplit(" = ", 1)
    right = right.strip()
    syscall, args = _split_syscall_and_args(left)
    row["kind"] = "syscall"
    row["syscall"] = syscall
    row["arguments"] = args

    m = _ERRNO_RE.match(right)
    if m:
        row["result_raw"] = right
        row["errno"] = m.group(2)
        row["errno_message"] = m.group(3)
        return row

    if _INT_RE.match(right):
        row["result_raw"] = right
        row["errno"] = None
        row["errno_message"] = None
        return row

    row["result_raw"] = right
    row["errno"] = None
    row["errno_message"] = None
    return row


def log_to_dataframe(path):
    text = path.read_text(encoding="utf-8", errors="replace")
    rows = [parse_strace_line(line) for line in text.splitlines()]
    df = pd.DataFrame(rows)
    df.insert(0, "line", range(1, len(df) + 1))
    return df


def filter_lines_under_prefix(df, prefix):
    if not prefix:
        return df
    mask = df["raw"].astype(str).str.contains(prefix, regex=False, na=False)
    return df.loc[mask].reset_index(drop=True)

In [ ]:
LOG_PATH = Path.cwd() / "out.log"
if not LOG_PATH.is_file():
    raise FileNotFoundError(
        "out.log not found. cd to the folder that contains it, or edit LOG_PATH. "
        f"Tried: {LOG_PATH.resolve()}"
    )

PATH_PREFIX = DEFAULT_PATH_PREFIX

df_all = log_to_dataframe(LOG_PATH)
df = filter_lines_under_prefix(df_all, PATH_PREFIX)
print(f"Rows: {len(df_all)} total → {len(df)} with prefix {PATH_PREFIX!r}")

Rows: 2862 total → 1495 with prefix '/home/anurag/opensource/TEST/'


In [7]:
from IPython.display import display

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

display(df)

,line,raw,kind,syscall,arguments,result_raw,errno,errno_message
0,1199,"getcwd(""/home/anurag/opensource/TEST/dulwich-1.1.0"", 512) = 43",syscall,getcwd,"""/home/anurag/opensource/TEST/dulwich-1.1.0"", 512",43,NaN,NaN
1,1203,"access(""/home/anurag/opensource/TEST/dulwich-1.1.0/objects"", F_OK) = -1 ENOENT (No such file or directory)",syscall,access,"""/home/anurag/opensource/TEST/dulwich-1.1.0/objects"", F_OK",-1 ENOENT (No such file or directory),ENOENT,No such file or directory
2,1204,"access(""/home/anurag/opensource/TEST/dulwich-1.1.0/.git/objects"", F_OK) = 0",syscall,access,"""/home/anurag/opensource/TEST/dulwich-1.1.0/.git/objects"", F_OK",0,NaN,NaN
3,1205,"access(""/home/anurag/opensource/TEST/dulwich-1.1.0/.git/objects"", F_OK) = 0",syscall,access,"""/home/anurag/opensource/TEST/dulwich-1.1.0/.git/objects"", F_OK",0,NaN,NaN
4,1206,"openat(AT_FDCWD, ""/home/anurag/opensource/TEST/dulwich-1.1.0/.git"", O_RDONLY|O_CLOEXEC) = 3",syscall,openat,"AT_FDCWD, ""/home/anurag/opensource/TEST/dulwich-1.1.0/.git"", O_RDONLY|O_CLOEXEC",3,NaN,NaN
5,1207,"openat(AT_FDCWD, ""/home/anurag/opensource/TEST/dulwich-1.1.0/.git/commondir"", O_RDONLY|O_CLOEXEC) = -1 ENOENT (No su...",syscall,openat,"AT_FDCWD, ""/home/anurag/opensource/TEST/dulwich-1.1.0/.git/commondir"", O_RDONLY|O_CLOEXEC",-1 ENOENT (No such file or directory),ENOENT,No such file or directory
6,1208,"openat(AT_FDCWD, ""/home/anurag/opensource/TEST/dulwich-1.1.0/.git/config"", O_RDONLY|O_CLOEXEC) = 3",syscall,openat,"AT_FDCWD, ""/home/anurag/opensource/TEST/dulwich-1.1.0/.git/config"", O_RDONLY|O_CLOEXEC",3,NaN,NaN
7,1209,"openat(AT_FDCWD, ""/home/anurag/opensource/TEST/dulwich-1.1.0/.git/commondir"", O_RDONLY|O_CLOEXEC) = -1 ENOENT (No su...",syscall,openat,"AT_FDCWD, ""/home/anurag/opensource/TEST/dulwich-1.1.0/.git/commondir"", O_RDONLY|O_CLOEXEC",-1 ENOENT (No such file or directory),ENOENT,No such file or directory
8,1226,"openat(AT_FDCWD, ""/home/anurag/opensource/TEST/dulwich-1.1.0/.git/commondir"", O_RDONLY|O_CLOEXEC) = -1 ENOENT (No su...",syscall,openat,"AT_FDCWD, ""/home/anurag/opensource/TEST/dulwich-1.1.0/.git/commondir"", O_RDONLY|O_CLOEXEC",-1 ENOENT (No such file or directory),ENOENT,No such file or directory
9,1227,"openat(AT_FDCWD, ""/home/anurag/opensource/TEST/dulwich-1.1.0/.git/HEAD"", O_RDONLY|O_CLOEXEC) = 3",syscall,openat,"AT_FDCWD, ""/home/anurag/opensource/TEST/dulwich-1.1.0/.git/HEAD"", O_RDONLY|O_CLOEXEC",3,NaN,NaN
